In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
from qforge.circuit import Qubit
from qforge.gates import CNOT, RY
import time
import numpy as np
import torch

torch.manual_seed(42)
num_samples = 40
wf = Qubit(2)

X = torch.rand(num_samples, 2) * 2 * math.pi
y = (X.sum(dim=1) > math.pi).float().view(-1, 1)

N_PARAMS = 2

def run_quantum_circuit(x_val, theta_val):
    wf = Qubit(2)
    RY(wf, 0, x_val[0]) 
    RY(wf, 1, x_val[1])
    CNOT(wf, 0, 1)
    RY(wf, 0, theta_val[0])
    RY(wf, 1, theta_val[1])
    probs = wf.probabilities()
    expval_z0 = probs[0] + probs[1] - probs[2] - probs[3]
    return float(expval_z0)

wf.visual_circuit()

|Q_0> : -M
          
|Q_1> : -M
          


In [13]:
class QforgeLayerOp(torch.autograd.Function):
    @staticmethod
    def forward(ctx, inputs, weights):
        ctx.save_for_backward(inputs, weights)
        return torch.tensor([run_quantum_circuit(inputs.tolist(), weights.tolist())],
                            dtype=torch.float32)

    @staticmethod
    def backward(ctx, grad_output):
        inputs, weights = ctx.saved_tensors
        x_val = inputs.tolist(); theta_val = weights.tolist()
        shift = math.pi / 2.0
        grads = []
        for i in range(len(theta_val)):
            tr = theta_val.copy(); tr[i] += shift
            tl = theta_val.copy(); tl[i] -= shift
            grads.append(0.5 * (run_quantum_circuit(x_val, tr) - run_quantum_circuit(x_val, tl)))
        return None, grad_output * torch.tensor(grads, dtype=torch.float32)

class QforgeQNN(nn.Module):
    """Hybrid: quantum layer (4 params) → classical Linear(1,1) → Sigmoid."""
    def __init__(self):
        super().__init__()
        self.theta = nn.Parameter(torch.rand(N_PARAMS) * 2 * math.pi)
        self.classical = nn.Linear(1, 1)

    def forward(self, x_batch):
        q = torch.stack([QforgeLayerOp.apply(x, self.theta) for x in x_batch])  # (N,1) ∈ [-1,1]
        return torch.sigmoid(self.classical(q))  # (N,1) ∈ [0,1]

model = QforgeQNN()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.1)

epochs = 100
print("  Qforge QNN Benchmark (2 qubits, 2 CNOT, 4 params + classical Linear)")
start_total = time.time()

for epoch in range(epochs):
        start_epoch = time.time()

        optimizer.zero_grad()
        preds = model(X)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()
        epoch_time = time.time() - start_epoch
        acc = ((preds > 0.5).float() == y).float().mean().item()
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:02d}/{epochs} | "
                  f"Loss: {loss.item():.4f} | Acc: {acc*100:5.1f}% | "
                  f"Time/epoch: {epoch_time:.2f}s")

total_time = time.time() - start_total

print(f"Tổng chạy Qforge: {total_time:.2f} giây "
          f"- (Tốc độ trung bình: {total_time/epochs:.2f} s/epoch)")

  Qforge QNN Benchmark (2 qubits, 2 CNOT, 4 params + classical Linear)
Epoch 01/100 | Loss: 0.3165 | Acc:  25.0% | Time/epoch: 0.01s
Epoch 05/100 | Loss: 0.2230 | Acc:  62.5% | Time/epoch: 0.00s
Epoch 10/100 | Loss: 0.1502 | Acc:  72.5% | Time/epoch: 0.00s
Epoch 15/100 | Loss: 0.1085 | Acc:  95.0% | Time/epoch: 0.00s
Epoch 20/100 | Loss: 0.0864 | Acc:  90.0% | Time/epoch: 0.00s
Epoch 25/100 | Loss: 0.0772 | Acc:  90.0% | Time/epoch: 0.00s
Epoch 30/100 | Loss: 0.0736 | Acc:  90.0% | Time/epoch: 0.00s
Epoch 35/100 | Loss: 0.0714 | Acc:  90.0% | Time/epoch: 0.00s
Epoch 40/100 | Loss: 0.0695 | Acc:  90.0% | Time/epoch: 0.00s
Epoch 45/100 | Loss: 0.0678 | Acc:  90.0% | Time/epoch: 0.00s
Epoch 50/100 | Loss: 0.0665 | Acc:  90.0% | Time/epoch: 0.00s
Epoch 55/100 | Loss: 0.0657 | Acc:  90.0% | Time/epoch: 0.00s
Epoch 60/100 | Loss: 0.0651 | Acc:  92.5% | Time/epoch: 0.00s
Epoch 65/100 | Loss: 0.0646 | Acc:  95.0% | Time/epoch: 0.00s
Epoch 70/100 | Loss: 0.0640 | Acc:  95.0% | Time/epoch: 0.00s